# Module 05 Lab - Data Preparation

**Objective:** To learn and apply the most common data preparation techniques. Raw data is rarely ready for a machine learning model. This process, also called preprocessing, is one of the most critical steps in the entire ML workflow.

**In this lab, you will write more of the code.** Read the explanations and then complete the tasks in the code cells.

## Part 1: Setup and Initial Look

We will continue using the Titanic dataset because it has the exact problems we need to solve: missing values and non-numeric data.

In [12]:
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Let's look at the missing values
print("--- Missing Values Before Cleaning ---")
print(df.isnull().sum())

--- Missing Values Before Cleaning ---
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


## Part 2: Handling Missing Values (Imputation)

**Concept:** Most machine learning models cannot handle missing values (`NaN`). We must deal with them. Dropping the rows is an option, but you lose data. A better way is **imputation**, which means filling in the missing values with a calculated guess.

Common imputation strategies:
*   **Mean:** Fill with the average value. Good for normally distributed data.
*   **Median:** Fill with the middle value. Better for skewed data or data with outliers (like `Fare`).
*   **Mode:** Fill with the most frequent value. Used for categorical data.

### Task 1: Impute the 'Age' Column

The 'Age' column is missing many values. Since age can be skewed (e.g., by a few very old passengers), using the **median** is a robust choice.

**Your Task:** Calculate the median of the 'Age' column and use the `.fillna()` method to replace the missing values.

In [13]:
# 1. Calculate the median of the 'Age' column
median_age = df['Age'].median()
print(f"Calculated median age: {median_age:.2f}")

# 2. Fill the missing values in 'Age' with the median safely without warnings
df['Age'] = df['Age'].fillna(median_age)

# 3. Verify that there are no more missing values in 'Age'
print("Missing values in 'Age' after imputation:")
print(df['Age'].isnull().sum())


Calculated median age: 28.00
Missing values in 'Age' after imputation:
0


## Part 3: Encoding Categorical Features

**Concept:** Machine learning models are mathematical, so they need numbers, not text. We need to convert categorical columns (like 'Sex' and 'Embarked') into a numerical format. The most common method is **One-Hot Encoding**.

One-Hot Encoding takes a column with `N` categories and turns it into `N` new columns, each with a `1` or `0`. For example, the 'Sex' column (`male`, `female`) becomes two new columns: `Sex_male` and `Sex_female`.

Pandas has a convenient function called `pd.get_dummies()` that does this for us.

### Task 2: One-Hot Encode Categorical Columns

**Your Task:** Use `pd.get_dummies()` to encode the 'Sex' and 'Embarked' columns. Make sure to drop the original columns after encoding.

In [14]:
# 1. Use get_dummies to create new columns for 'Sex' and 'Embarked'
# Set `drop_first=True` to avoid multicollinearity (a statistical issue)
encoded_df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

# 2. Display the first few rows of the new DataFrame to see the new columns
print(encoded_df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name   Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris  22.0      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0      1      0   
2                             Heikkinen, Miss. Laina  26.0      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  35.0      1      0   
4                           Allen, Mr. William Henry  35.0      0      0   

             Ticket     Fare Cabin  Sex_male  Embarked_Q  Embarked_S  
0         A/5 21171   7.2500   NaN      True       False        True  
1          PC 17599  71.2833   C85     False       False       False  
2  STON/O2. 3101282   7.9250   NaN     False       False        True  
3            113803  53.1000  C123     Fal

In [15]:
# --- ENTER YOUR CODE HERE ---

# 1. Use get_dummies to create new columns for 'Sex' and 'Embarked'
#    Set `drop_first=True` to avoid multicollinearity (a statistical issue), which drops one of the new columns (e.g., just having `Sex_male` is enough to know if someone is female).
# encoded_df = pd.get_dummies(df, columns=[... , ...], drop_first=True)

# 2. Display the first few rows of the new DataFrame to see the new columns
# print(encoded_df.head())

## Part 4: Feature Scaling

**Concept:** Many models are sensitive to the scale of the features. For example, `Age` (from 0-80) and `Fare` (from 0-512) are on very different scales. This can cause the model to incorrectly believe that `Fare` is a more important feature simply because its values are larger.

**Feature Scaling** solves this by putting all features on a similar scale. A common method is **Standardization** (`StandardScaler` in scikit-learn), which rescales the data to have a mean of 0 and a standard deviation of 1.

**Important:** You only scale your numerical features, not your target variable or your newly encoded categorical columns.

### Task 3: Scale the 'Age' and 'Fare' Columns

**Your Task:** Use `StandardScaler` from `sklearn.preprocessing` to scale the 'Age' and 'Fare' columns.

In [16]:
from sklearn.preprocessing import StandardScaler

# 1. Create an instance of the StandardScaler
scaler = StandardScaler()

# 2. Select the columns to scale
columns_to_scale = ['Age', 'Fare']

# 3. Fit the scaler to the data and transform it
# Using the encoded_df passed forward from Jonathan's One-Hot Encoding step
encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
print(encoded_df.head())


   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name       Age  SibSp  Parch  \
0                            Braund, Mr. Owen Harris -0.565736      1      0   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  0.663861      1      0   
2                             Heikkinen, Miss. Laina -0.258337      0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  0.433312      1      0   
4                           Allen, Mr. William Henry  0.433312      0      0   

             Ticket      Fare Cabin  Sex_male  Embarked_Q  Embarked_S  
0         A/5 21171 -0.502445   NaN      True       False        True  
1          PC 17599  0.786845   C85     False       False       False  
2  STON/O2. 3101282 -0.488854   NaN     False       False        True  
3            1

In [17]:
from sklearn.preprocessing import StandardScaler

# --- ENTER YOUR CODE HERE ---

# 1. Create an instance of the StandardScaler
# scaler = ...

# 2. Select the columns to scale
# columns_to_scale = ['Age', 'Fare']

# 3. Fit the scaler to the data and transform it
#    Note: We are using the `encoded_df` from the previous step if you created it.
# encoded_df[columns_to_scale] = scaler.fit_transform(encoded_df[columns_to_scale])

# 4. Display the first few rows to see the scaled data
# print(encoded_df.head())

## 📝 Knowledge Check

**Instructions:** Answer the following questions in this markdown cell.

### 1. Why is it often better to impute missing values with the median instead of the mean?

**Group Analytical Insights (Contributed by Dana Bolden & Astríd Lorenzana):**

* **Astríd's Analysis:** Compared to the mean, the median is less sensitive to outliers. The mean can have a huge change in the way it is skewed if a really high or low value was obtained. The median is more balanced and less susceptive to drastic changes from an outlier.
* **Dana's Statistical Expansion:** Outliers and extreme values have a strong, distorting impact on the mathematical mean (average) of a dataset. If the continuous distribution of a feature is highly skewed—such as the Titanic passenger fares, which contain a massive volume of low-value tickets combined with a few exceptionally high luxury-suite values—the mean will be pulled significantly higher than what represents a "typical" passenger.

**Conclusion:** The median, representing the absolute middle value in an ordered sequence, is inherently resistant to these anomalies. Choosing the median provides a statistically reasonable estimation of the true center of data density without being artificially inflated or deflated.


---

### 2. Explain in your own words what One-Hot Encoding does and why it is necessary.

One-Hot Encoding is a technique used to transform categorical textual information (such as a passenger's port of embarkation or gender) into a strictly numerical, binary format. It does this by converting one column of categories into multiple new columns, one for each unique category, with 1's (True) and 0's (False) in each. This is because the majority of machine learning algorithms work with mathematical equations and are unable to work with plain text. In addition, by arbitrarily assigning numbers to the categories (Cherbourg: 1, Queenstown: 2, Southampton: 3), the algorithm could potentially misread a mathematical hierarchy (Queenstown > Cherbourg > Southampton). One Hot Encoding ensures this doesn't result in this false ranking because it considers each category as an equal feature.

---

### 3. Would you need to apply Feature Scaling to a Decision Tree model? Why or why not? (Hint: Think about how a Decision Tree makes its splits).

No, for Decision Tree models, feature scaling (standardization or normalization) is not required. Decision Trees differ from such algorithms as K-Nearest Neighbors or Support Vector Machines, which calculate the mathematical distance or geometry between data points, in that Decision Trees predict by evaluating simple, rule-based splits on individual features one at a time (e.g., "Is Age > 30?" or "Is Fare < 50?"). Scaling a feature will only alter its numeric range, but will not alter the relative distribution or sequencing of the data points, so the same split points will be found for the feature when scaled compared to when unscaled.


In [18]:
# Experiment 1: Calculate the exact survival rate based on gender
# This helps verify why 'Sex' is such a critical feature for our ML model
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100

print("--- Experiment 1: Survival Rate by Gender ---")
print(f"Female Survival Rate: {survival_by_sex['female']:.2f}%")
print(f"Male Survival Rate: {survival_by_sex['male']:.2f}%")


--- Experiment 1: Survival Rate by Gender ---
Female Survival Rate: 74.20%
Male Survival Rate: 18.89%


In [19]:
# Experiment 2: Check the average ticket price for each passenger class
# This illustrates the extreme variance and outliers in the 'Fare' column
average_fare_by_class = df.groupby('Pclass')['Fare'].mean()

print("--- Experiment 2: Average Fare by Passenger Class ---")
for pclass, fare in average_fare_by_class.items():
    print(f"Class {pclass}: ${fare:.2f}")


--- Experiment 2: Average Fare by Passenger Class ---
Class 1: $84.15
Class 2: $20.66
Class 3: $13.68


In [20]:
# Experiment 3: Compare raw data variance vs. scaled data variance
# This directly proves that Ruben's Feature Scaling successfully normalized the scales
print("--- Experiment 3: Scaling Verification ---")
print(f"Raw 'Fare' Max Value (Before): {df['Fare'].max():.2f}")
print(f"Scaled 'Fare' Max Value (After): {encoded_df['Fare'].max():.2f}")
print(f"Scaled 'Fare' Mean (Target is ~0): {encoded_df['Fare'].mean():.2f}")


--- Experiment 3: Scaling Verification ---
Raw 'Fare' Max Value (Before): 512.33
Scaled 'Fare' Max Value (After): 9.67
Scaled 'Fare' Mean (Target is ~0): 0.00


# Reflection
- I do not know how to code so when I got started on this lab it is pretty overwhelming. Upon looking at resources, I noticed that even though the variables the person used were different the execution is meant to work the same. It made me feel relieved that I am not a lost case and it encourages me a bit to take some initiative after completing part 2 of this lab.
- When I received the output for part 2, there was a note that stated that for future reference, I should tweak the statement I made for any upcoming panda, which lets me keep in mind that I should try to constantly improve.

==================================================================
LAB 05 CONTRIBUTION JOURNAL: DATA PREPARATION
Course Code: ITAI1371
Group Name: 01
==================================================================

1. MEMBER LOG & TASKS COMPLETED:

* Student Name: Cuong Dang
  - Tasks: Acted as the Master Compiler to assemble all independent fragments into the final synchronized Jupyter Notebook and managed the GitHub repository pipeline. Modernized and debugged the baseline code by replacing deprecated 'inplace=True' parameters to eliminate critical system FutureWarnings. Designed and programmed 3 additional exploratory data experiments (Survival by Sex, Fare by Pclass, and Scale Variance Verification) to expand the analytical depth of the lab.

* Student Name: Astríd Lorenzana
  - Tasks: Ran Part 1 of the code and wrote the baseline implementation code for Part 2. Answered Knowledge Check Question 1 regarding the impact of skewed distributions and outliers on mean metrics. Authored the qualitative personal text for the final Lab Reflection section.

* Student Name: Jonathan Aldana
  - Tasks: Wrote the implementation code for Part 3 (Task 2: One-Hot Encoding). Programmed the pd.get_dummies() framework using the 'drop_first=True' configuration to eliminate potential multicollinearity issues within the feature matrix.

* Student Name: Ruben Nuno
  - Tasks: Wrote the implementation code for Part 4 (Task 3: Feature Scaling). Initialized the StandardScaler module from sklearn.preprocessing and executed the fit_transform operation on continuous numeric variables ('Age' and 'Fare').

* Student Name: Dana Bolden
  - Tasks: Completed the comprehensive academic and theoretical analysis for the remaining Knowledge Check sections. Authored the precise, verbatim technical responses for Question 2 (categorical binary vector transformation mapping) and Question 3 (Decision Tree orthogonal partition mechanics).

2. TEAMWORK DISCLOSURE:
Due to internal group coordination challenges and communication overhead, consolidation was performed asynchronously. However, every single group member contributed unique, distinct technical or analytical assets to this final deliverable. We respectfully request grading evaluations to reflect these specific boundaries.
==================================================================
